# CUDA Optimization Course - Module 5: DataLoader Optimization

Optimize data loading pipeline with pin_memory, num_workers, and batch size strategies.

## Setup

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time
import psutil
import os

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")
print(f"CPU cores: {os.cpu_count()}")

Device: mps
CPU cores: 16


## Create Dummy Dataset

In [2]:
# Create a larger dataset to simulate real-world scenario
num_samples = 10000
seq_length = 256
vocab_size = 10000

# Generate data on CPU first
input_ids = torch.randint(0, vocab_size, (num_samples, seq_length))
labels = torch.randint(0, vocab_size, (num_samples, seq_length))

dataset = TensorDataset(input_ids, labels)
print(f"Dataset size: {len(dataset)} samples")
print(f"Each sample: input {input_ids[0].shape}, label {labels[0].shape}")

Dataset size: 10000 samples
Each sample: input torch.Size([256]), label torch.Size([256])


## Simple Model for Benchmarking

In [3]:
class SimpleModel(nn.Module):
    def __init__(self, vocab_size=10000, d_model=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x).mean(dim=1)  # Average pooling
        x = self.fc(x)
        return x

model = SimpleModel().to(device)

## Baseline: No Optimization

In [4]:
batch_size = 64

# Baseline: default settings
dataloader_baseline = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Baseline DataLoader (num_workers=0, pin_memory=False):")

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

total_samples = 0
for batch_idx, (input_ids, labels) in enumerate(dataloader_baseline):
    input_ids = input_ids.to(device)
    labels = labels.to(device)
    output = model(input_ids)
    total_samples += input_ids.shape[0]

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
baseline_time = time.perf_counter() - start

print(f"Time: {baseline_time:.2f}s")
print(f"Throughput: {total_samples / baseline_time:.0f} samples/sec")

Baseline DataLoader (num_workers=0, pin_memory=False):
Time: 0.50s
Throughput: 19814 samples/sec


## Optimization 1: pin_memory (CUDA only)

In [5]:
if device == 'cuda':
    dataloader_pinned = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True  # Pin memory for faster CPU->GPU transfer
    )
    
    print("\nOptimized DataLoader (pin_memory=True):")
    
    torch.cuda.synchronize()
    start = time.perf_counter()
    
    total_samples = 0
    for batch_idx, (input_ids, labels) in enumerate(dataloader_pinned):
        input_ids = input_ids.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        output = model(input_ids)
        total_samples += input_ids.shape[0]
    
    torch.cuda.synchronize()
    pinned_time = time.perf_counter() - start
    
    print(f"Time: {pinned_time:.2f}s")
    print(f"Throughput: {total_samples / pinned_time:.0f} samples/sec")
    print(f"Speedup: {baseline_time / pinned_time:.2f}x")
else:
    print("\npin_memory is CUDA-specific. Skipping on", device)


pin_memory is CUDA-specific. Skipping on mps


## Optimization 2: num_workers (Parallel Data Loading)

In [6]:
# Test different num_workers values
num_workers_options = [0, 2, 4, 6]
results = {}

for num_workers in num_workers_options:
    dataloader_workers = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=min(num_workers, os.cpu_count()),
        pin_memory=device == 'cuda'  # Use pin_memory if CUDA
    )
    
    torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
    start = time.perf_counter()
    
    total_samples = 0
    for batch_idx, (input_ids, labels) in enumerate(dataloader_workers):
        input_ids = input_ids.to(device, non_blocking=device == 'cuda')
        labels = labels.to(device, non_blocking=device == 'cuda')
        output = model(input_ids)
        total_samples += input_ids.shape[0]
    
    torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
    elapsed = time.perf_counter() - start
    results[num_workers] = elapsed
    
    print(f"num_workers={num_workers}: {elapsed:.2f}s ({total_samples / elapsed:.0f} samples/sec)")

print(f"\nBest num_workers: {min(results, key=results.get)} (Speedup: {baseline_time / min(results.values()):.2f}x)")

num_workers=0: 0.23s (43666 samples/sec)
num_workers=2: 1.31s (7606 samples/sec)
num_workers=4: 0.91s (11036 samples/sec)
num_workers=6: 0.96s (10468 samples/sec)

Best num_workers: 0 (Speedup: 2.20x)


## Optimization 3: Batch Size Tuning

In [7]:
batch_sizes = [16, 32, 64, 128, 256]
batch_results = {}

best_num_workers = min(results, key=results.get)

for bs in batch_sizes:
    dataloader_bs = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=False,
        num_workers=best_num_workers,
        pin_memory=device == 'cuda'
    )
    
    torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
    start = time.perf_counter()
    
    total_samples = 0
    for batch_idx, (input_ids, labels) in enumerate(dataloader_bs):
        input_ids = input_ids.to(device, non_blocking=device == 'cuda')
        labels = labels.to(device, non_blocking=device == 'cuda')
        output = model(input_ids)
        total_samples += input_ids.shape[0]
    
    torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
    elapsed = time.perf_counter() - start
    batch_results[bs] = elapsed
    
    print(f"Batch size {bs:3d}: {elapsed:.2f}s ({total_samples / elapsed:.0f} samples/sec)")

Batch size  16: 0.48s (20836 samples/sec)
Batch size  32: 0.24s (41420 samples/sec)
Batch size  64: 0.12s (83036 samples/sec)
Batch size 128: 0.15s (67628 samples/sec)
Batch size 256: 0.07s (148785 samples/sec)


## Optimization 4: Combined Best Practices

In [10]:
best_batch_size = min(batch_results, key=batch_results.get)

dataloader_optimized = DataLoader(
    dataset,
    batch_size=best_batch_size,
    shuffle=False,
    num_workers=best_num_workers,
    pin_memory=device == 'cuda',
    prefetch_factor=2 if best_num_workers > 0 else None,  # Prefetch batches
    persistent_workers=best_num_workers > 0  # Keep workers alive
)

print("\n=== Combined Optimization ===")
print(f"Batch size: {best_batch_size}")
print(f"num_workers: {best_num_workers}")
print(f"pin_memory: {device == 'cuda'}")

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

total_samples = 0
for batch_idx, (input_ids, labels) in enumerate(dataloader_optimized):
    input_ids = input_ids.to(device, non_blocking=device == 'cuda')
    labels = labels.to(device, non_blocking=device == 'cuda')
    output = model(input_ids)
    total_samples += input_ids.shape[0]

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
optimized_time = time.perf_counter() - start

print(f"\nTime: {optimized_time:.2f}s")
print(f"Throughput: {total_samples / optimized_time:.0f} samples/sec")
print(f"\nSpeedup vs Baseline: {baseline_time / optimized_time:.2f}x")


=== Combined Optimization ===
Batch size: 256
num_workers: 0
pin_memory: False

Time: 0.14s
Throughput: 73790 samples/sec

Speedup vs Baseline: 3.72x


## DataLoader Configuration Guide

```python
# Optimal DataLoader configuration
dataloader = DataLoader(
    dataset,
    batch_size=64,              # Balance memory and throughput
    shuffle=True,                # Randomize for training
    num_workers=4,               # Parallel data loading
    pin_memory=True,             # CUDA: Fast CPU->GPU transfer
    prefetch_factor=2,           # Prefetch batches
    persistent_workers=True,     # Keep workers alive (fewer restarts)
    drop_last=False              # Drop incomplete batch
)

# Training loop with non_blocking
for input_ids, labels in dataloader:
    input_ids = input_ids.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    output = model(input_ids)
```

## Key Takeaways

1. **pin_memory**: 10-30% faster data transfer on CUDA (CPU->GPU)
2. **num_workers**: Parallel data loading from disk, use 4-8 workers
3. **Batch Size**: Larger batches improve throughput but use more memory
4. **prefetch_factor**: Prefetch batches to avoid stalls
5. **persistent_workers**: Reduce overhead by keeping worker processes alive
6. **non_blocking=True**: Overlap data transfer with computation
7. **M3 MacBook**: num_workers has less impact on Mac, focus on batch size